# Eksperimen: Deteksi Penyakit Daun Anggur
## Metode: Tangential Direction (TD) + K-Nearest Neighbors (KNN)

**Deskripsi:** Notebook ini mengimplementasikan pipeline penuh dari preprocessing hingga evaluasi model untuk klasifikasi penyakit daun anggur menggunakan fitur Tangential Direction dan KNN.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score, learning_curve
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
from tqdm.notebook import tqdm

sns.set_style('whitegrid')
%matplotlib inline

print('Libraries loaded successfully')

---
## 1. Definisikan Class Names & Path

In [ ]:
CLASS_NAMES = ['Black_Rot', 'ESCA', 'Leaf_Blight', 'Healthy']
CLASS_LABELS = {name: i for i, name in enumerate(CLASS_NAMES)}

DATA_DIR = Path('../data/train')
MODEL_DIR = Path('../models')
RESULT_DIR = Path('../results')
MODEL_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

print(f'Data: {DATA_DIR}')
print(f'Classes: {CLASS_NAMES}')
for cls in CLASS_NAMES:
    path = DATA_DIR / cls
    if path.exists():
        files = list(path.glob('*'))
        print(f'  {cls}: {len(files)} images')

---
## 2. Load & Preprocessing Pipeline

Langkah preprocessing:
1. Resize ke 256x256
2. Median filter (denoising)
3. CLAHE (contrast enhancement)
4. Otsu thresholding untuk segmentasi daun
5. Ekstraksi kontur utama

In [ ]:
from src.preprocess import preprocess_pipeline, create_mask, get_main_contour

sample_path = list((DATA_DIR / CLASS_NAMES[0]).glob('*.jpg'))[0]
img_original = cv2.imread(str(sample_path))
img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)
img_processed = preprocess_pipeline(sample_path)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_original)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(img_processed)
axes[1].set_title('After Preprocessing')
axes[1].axis('off')

img_gray = cv2.cvtColor(img_processed, cv2.COLOR_RGB2GRAY)
mask = create_mask(img_gray)
contour = get_main_contour(mask)
img_contour = img_processed.copy()
cv2.drawContours(img_contour, [contour], -1, (255, 0, 0), 2)
axes[2].imshow(img_contour)
axes[2].set_title('Contour (Edge Detection)')
axes[2].axis('off')
plt.tight_layout()
plt.show()

---
## 3. Ekstraksi Fitur Tangential Direction

Fitur yang diekstrak:
1. **TD Histogram** (36 bins) — distribusi arah tangensial kontur daun
2. **Color Histogram** (9×4 = 36 bins) — H, S, V, A channel
3. **Texture Features** (4) — GLCM contrast, energy, homogeneity, correlation

In [ ]:
from src.features import (
    compute_tangent_angles, histogram_tangent_direction,
    extract_td_features, extract_color_features,
    extract_texture_features, extract_all_features, FEATURE_NAMES
)

td_feat = extract_td_features(img_gray)
color_feat = extract_color_features(img_processed)
texture_feat = extract_texture_features(img_gray)
all_feat = extract_all_features(img_processed)

print(f'TD features: {len(td_feat)} bins')
print(f'Color features: {len(color_feat)} values')
print(f'Texture features: {len(texture_feat)} values')
print(f'Total feature vector: {len(all_feat)} dimensions')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(len(td_feat)), td_feat)
axes[0].set_title('Histogram of Tangent Directions (36 bins)')
axes[0].set_xlabel('Angle bin')
axes[0].set_ylabel('Frequency')
axes[1].bar(range(len(texture_feat)), texture_feat)
axes[1].set_title('GLCM Texture Features')
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(['Contrast', 'Energy', 'Homogeneity', 'Correlation'])
plt.tight_layout()
plt.show()

### Visualisasi Perbandingan TD Antar Kelas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, cls_name in enumerate(CLASS_NAMES):
    cls_dir = DATA_DIR / cls_name
    samples = list(cls_dir.glob('*.jpg'))[:5]
    td_hists = []
    for sp in samples:
        img = preprocess_pipeline(sp)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        td = extract_td_features(gray)
        if td is not None:
            td_hists.append(td)
    if td_hists:
        mean_hist = np.mean(td_hists, axis=0)
        row, col = divmod(idx, 2)
        axes[row][col].bar(range(36), mean_hist)
        axes[row][col].set_title(f'{cls_name} - Mean TD Histogram')
        axes[row][col].set_xlabel('Direction bin')
        axes[row][col].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

---
## 4. Bangun Dataset Fitur

In [ ]:
from src.train import build_dataset

X, y = build_dataset(DATA_DIR)
print(f'Dataset shape: {X.shape}')
print(f'Labels shape: {y.shape}')
unique, counts = np.unique(y, return_counts=True)
for cls_idx, count in zip(unique, counts):
    print(f'  {CLASS_NAMES[cls_idx]}: {count} samples')

---
## 5. Split Data (70:15:15)

In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(X, y))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx2, val_idx = next(sss_val.split(X_train, y_train))
X_train_final, X_val = X_train[train_idx2], X_train[val_idx]
y_train_final, y_val = y_train[train_idx2], y_train[val_idx]

print(f'Train: {len(X_train_final)} samples')
print(f'Validation: {len(X_val)} samples')
print(f'Test: {len(X_test)} samples')

---
## 6. GridSearchCV — Hyperparameter Tuning KNN

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

grid = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='accuracy',
    n_jobs=-1, verbose=1
)
grid.fit(X_train_final, y_train_final)

print(f'\nBest parameters: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')
print(f'Validation accuracy: {grid.score(X_val, y_val):.4f}')

In [ ]:
results_df = pd.DataFrame(grid.cv_results_)
pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_knn__n_neighbors',
    columns=['param_knn__weights', 'param_knn__p']
)
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd')
plt.title('Grid Search Results - Mean CV Accuracy')
plt.tight_layout()
plt.show()

---
## 7. Evaluasi Model pada Test Set

In [ ]:
y_pred = grid.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {acc:.4f}\n')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Learning Curves & Cross-Validation

In [ ]:
scaler = grid.best_estimator_.named_steps['scaler']
X_scaled = scaler.transform(X)
best_knn = grid.best_estimator_.named_steps['knn']

cv_scores = cross_val_score(grid.best_estimator_, X, y, cv=5, scoring='accuracy')
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), cv_scores, color='skyblue', edgecolor='navy')
plt.axhline(y=np.mean(cv_scores), color='red', linestyle='--',
            label=f'Mean = {np.mean(cv_scores):.3f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('5-Fold Cross-Validation Scores')
plt.ylim(0, 1)
plt.legend()
plt.savefig(RESULT_DIR / 'cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'CV Scores: {cv_scores}')
print(f'Mean CV: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}')

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    grid.best_estimator_, X, y, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_mean - train_std,
                 train_mean + train_std, alpha=0.1, color='blue')
plt.fill_between(train_sizes, val_mean - val_std,
                 val_mean + val_std, alpha=0.1, color='orange')
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
plt.plot(train_sizes, val_mean, 'o-', color='orange', label='Cross-validation score')
plt.title('Learning Curves')
plt.xlabel('Training examples')
plt.ylabel('Accuracy')
plt.legend(loc='best')
plt.grid(True)
plt.savefig(RESULT_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Robustness Test

Menguji performa model terhadap:
- Gaussian noise (σ = 0.01, 0.05, 0.1)
- Brightness variation (×0.5, ×0.75, ×1.25, ×1.5)

In [ ]:
from src.evaluate import robustness_test
robustness_test(grid, X_test, y_test, RESULT_DIR)

---
## 10. Simpan Model & Hasil

In [ ]:
model_path = MODEL_DIR / 'knn_model.pkl'
joblib.dump(grid, model_path)
np.save(RESULT_DIR / 'X_test.npy', X_test)
np.save(RESULT_DIR / 'y_test.npy', y_test)
np.save(RESULT_DIR / 'y_pred.npy', y_pred)

print(f'Model saved: {model_path}')
print(f'Results saved: {RESULT_DIR}/')

---
## 11. Kesimpulan Eksperimen

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print('=' * 60)
print('  RINGKASAN HASIL EKSPERIMEN')
print('=' * 60)
print(f'\nMetode: Tangential Direction + K-Nearest Neighbors')
print(f'Dataset: Grape Disease Original ({len(X)} samples)')
print(f'Feature dimension: {X.shape[1]}')
print(f'Best KNN params: {grid.best_params_}')
print()
print(f'Test Accuracy:  {acc:.4f}')
print(f'Precision (weighted): {precision_score(y_test, y_pred, average="weighted"):.4f}')
print(f'Recall (weighted):    {recall_score(y_test, y_pred, average="weighted"):.4f}')
print(f'F1-Score (weighted):  {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'CV Score (5-fold):    {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}')